# Exploração de Dados

In [ ]:
# %% [markdown]
# # 🔐 Análise de Vulnerabilidades de Segurança
# 
# **Projeto para S4 System | Arquitetura Medalhão Bronze → Silver → Gold**

# %%
# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.etl_pipeline import SecurityAnalyticsETL
from src.analytics import SecurityAnalytics
from src.snowflake_simulator import demonstrate_snowpark_concepts

# Configurações
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

# %% [markdown]
# ## 1. Executar Pipeline ETL

# %%
# Executar pipeline completo
pipeline = SecurityAnalyticsETL()
pipeline.run_pipeline()

# %% [markdown]
# ## 2. Análise da Camada Silver

# %%
# Carregar dados processados
df_silver = pd.read_parquet("../data/processed/silver_vulnerabilities.parquet")

print(f"Shape: {df_silver.shape}")
print(f"Colunas: {list(df_silver.columns)}")
df_silver.head()

# %% [markdown]
# ## 3. Estatísticas Descritivas

# %%
# Resumo estatístico
df_silver.describe()

# %%
# Análise por nível de risco
df_silver.groupby('risk_level').agg({
    'cve_id': 'count',
    'score_cvss': ['mean', 'max'],
    'vendor_project': 'nunique'
})

# %% [markdown]
# ## 4. Visualizações

# %%
# Distribuição de scores por vendor
plt.figure(figsize=(12,6))
top_vendors = df_silver['vendor_project'].value_counts().head(10).index
df_top = df_silver[df_silver['vendor_project'].isin(top_vendors)]

sns.boxplot(data=df_top, x='vendor_project', y='score_cvss')
plt.xticks(rotation=45)
plt.title('Distribuição de Scores CVSS por Vendor')
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 5. Demonstração Snowpark

# %%
# Simular conceitos do Snowpark
demonstrate_snowpark_concepts(df_silver)

# %% [markdown]
# ## 6. Relatórios Gerenciais

# %%
# Gerar relatórios
analytics = SecurityAnalytics()
report = analytics.generate_executive_report(df_silver)
analytics.create_visualizations(df_silver)
analytics.generate_conformity_report(df_silver, "NIST")